# Dataset Summary

## Dataset Source

The selected dataset consists of experimental Li-ion battery aging data containing charge, discharge, and impedance measurements. The dataset contains four batteries:

- B0005
- B0006
- B0007
- B0018

## Number of Batteries

4 batteries.

## Operation Records

The dataset contains three types of operations:

- Charge
- Discharge
- Impedance

| Battery | Total Operations | Charge | Discharge | Impedance |
|---------|-----------------:|-------:|----------:|----------:|
| B0005 | 616 | 170 | 168 | 278 |
| B0006 | 616 | 170 | 168 | 278 |
| B0007 | 616 | 170 | 168 | 278 |
| B0018 | 319 | 134 | 132 | 53 |
| **Total** | **2167** | **644** | **636** | **887** |

## Discharge Data

Discharge operations contain the following measurements:

- `Voltage_measured`
- `Current_measured`
- `Temperature_measured`
- `Current_load`
- `Voltage_load`
- `Time`
- `Capacity`

All **636 discharge operations** contain a valid capacity measurement.

The measured capacity generally decreases over repeated discharge operations, indicating battery degradation with aging.

The observed initial and final discharge capacities are:

| Battery | Initial Capacity (Ah) | Final Capacity (Ah) | Minimum Capacity (Ah) |
|---------|----------------------:|--------------------:|----------------------:|
| B0005 | 1.8565 | 1.3251 | 1.2875 |
| B0006 | 2.0353 | 1.1857 | 1.1538 |
| B0007 | 1.8911 | 1.4325 | 1.4005 |
| B0018 | 1.8550 | 1.3411 | 1.3411 |

## Sampling Information

The time-series measurements are sampled at non-uniform time intervals.

The number of samples per discharge operation varies between operations:

| Battery | Minimum Samples | Maximum Samples | Average Samples |
|---------|----------------:|----------------:|----------------:|
| B0005 | 179 | 371 | 299.32 |
| B0006 | 179 | 371 | 299.32 |
| B0007 | 179 | 371 | 299.32 |
| B0018 | 200 | 366 | 264.14 |

Therefore, the raw discharge measurements are variable-length time series and will require appropriate standardization during preprocessing.

## Missing Values

No missing numerical values were detected in the charge, discharge, or impedance data of B0005, B0006, and B0007.

B0018 contains **6 NaN values in its charge operations**. No NaN values were found in its discharge or impedance operations.

These missing values will be investigated and handled during preprocessing. No data will be removed without documenting the reason.

## Data Quality Checks

The following checks were performed:

- No empty discharge time sequences were found.
- No non-monotonic discharge time sequences were found.
- No duplicate timestamps were found in discharge operations.
- All 636 discharge operations contain a valid capacity measurement.
- Discharge measurements are stored as numerical floating-point values.
- Discharge operations contain variable-length time-series measurements.
- Capacity values show an overall decreasing trend across battery aging.

## Relevant Battery Parameters

The relevant parameters identified in the dataset include:

- Battery ID
- Operation type
- Time
- Voltage
- Current
- Temperature
- Capacity
- Battery impedance
- Rectified impedance
- Electrolyte resistance (`Re`)
- Charge transfer resistance (`Rct`)

The final set of features to be included in the standardized dataset will be finalized in consultation with the other project members.

## Cycle Definition

The raw dataset contains charge, discharge, and impedance operations in a non-uniform sequence. Therefore, a cycle should not be identified simply by grouping every fixed number of raw records.

For the current analysis, discharge operations are treated as the primary cycle-level records because each discharge operation contains a measured capacity value. The final cycle definition and SOH formulation will be confirmed with the project team before the standardized dataset is created.

## Initial Dataset Findings — Battery B0005

The B0005 MATLAB file contains 616 operation records consisting of:

- 170 charge operations
- 168 discharge operations
- 278 impedance operations

Each operation contains four top-level fields:

- `type`
- `ambient_temperature`
- `time`
- `data`

For charge operations, the measurement fields are:

- `Voltage_measured`
- `Current_measured`
- `Temperature_measured`
- `Current_charge`
- `Voltage_charge`
- `Time`

For discharge operations, the observed fields in B0005 are:

- `Voltage_measured`
- `Current_measured`
- `Temperature_measured`
- `Current_load`
- `Voltage_load`
- `Time`
- `Capacity`

The first discharge operation has a measured capacity of approximately 1.8565 Ah and contains 197 samples for each time-series measurement.

The discharge current is approximately -2 A after the initial samples, while battery voltage decreases and temperature increases during the discharge operation.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import loadmat
from collections import Counter

print("Environment ready!")

Environment ready!


In [2]:
# BatteryIQ project paths

PROJECT_ROOT = Path.cwd().parent
RAW_PATH = PROJECT_ROOT / "data" / "raw"

print("Project root:")
print(PROJECT_ROOT)

print("\nRaw data path:")
print(RAW_PATH)

print("\nRaw data exists:", RAW_PATH.exists())

Project root:
d:\VIT Academic\TY\sem 5\EDI\BatteryIQ

Raw data path:
d:\VIT Academic\TY\sem 5\EDI\BatteryIQ\data\raw

Raw data exists: True


In [3]:
battery_files = [
    "B0005.mat",
    "B0006.mat",
    "B0007.mat",
    "B0018.mat"
]

print("Available battery files:\n")

for filename in battery_files:
    path = RAW_PATH / filename
    print(f"{filename}: {'Found' if path.exists() else 'NOT FOUND'}")

Available battery files:

B0005.mat: Found
B0006.mat: Found
B0007.mat: Found
B0018.mat: Found


In [4]:
all_batteries = {}

for filename in battery_files:
    battery_name = filename.replace(".mat", "")
    file_path = RAW_PATH / filename
    
    all_batteries[battery_name] = loadmat(file_path)
    
print("Loaded batteries:")
print(list(all_batteries.keys()))

Loaded batteries:
['B0005', 'B0006', 'B0007', 'B0018']


In [5]:
battery = all_batteries["B0005"]

print("Top-level keys:")
print(battery.keys())

cycles = battery["B0005"]["cycle"]

print("Cycle data type:", type(cycles))
print("Cycle data shape:", cycles.shape)
print("Cycle fields:", cycles.dtype.names)

Top-level keys:
dict_keys(['__header__', '__version__', '__globals__', 'B0005'])
Cycle data type: <class 'numpy.ndarray'>
Cycle data shape: (1, 1)
Cycle fields: None


In [6]:
operation_summary = {}

for battery_name, battery in all_batteries.items():
    
    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]
    
    cycle_types = []
    
    for i in range(cycle_data.shape[1]):
        cycle_type = cycle_data[0, i]["type"].item()
        cycle_types.append(cycle_type)
    
    counts = Counter(cycle_types)
    
    operation_summary[battery_name] = counts
    
    print(f"\n{battery_name}")
    print("-" * 30)
    print("Total operations:", len(cycle_types))
    print("Charge:", counts["charge"])
    print("Discharge:", counts["discharge"])
    print("Impedance:", counts["impedance"])


B0005
------------------------------
Total operations: 616
Charge: 170
Discharge: 168
Impedance: 278

B0006
------------------------------
Total operations: 616
Charge: 170
Discharge: 168
Impedance: 278

B0007
------------------------------
Total operations: 616
Charge: 170
Discharge: 168
Impedance: 278

B0018
------------------------------
Total operations: 319
Charge: 134
Discharge: 132
Impedance: 53


In [7]:
for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    counts = Counter()
    nan_counts = Counter()

    for i in range(cycle_data.shape[1]):

        cycle_type = cycle_data[0, i]["type"].item()
        counts[cycle_type] += 1

        data = cycle_data[0, i]["data"][0, 0]

        for field in data.dtype.names:
            values = np.asarray(data[field]).flatten()

            if np.issubdtype(values.dtype, np.number):
                nan_counts[cycle_type] += np.isnan(values).sum()

    print(f"\n{battery_name}")
    print("-" * 30)

    for operation in ["charge", "discharge", "impedance"]:
        print(
            f"{operation:10s}: "
            f"{counts[operation]} operations, "
            f"{nan_counts[operation]} NaN values"
        )


B0005
------------------------------
charge    : 170 operations, 0 NaN values
discharge : 168 operations, 0 NaN values
impedance : 278 operations, 0 NaN values

B0006
------------------------------
charge    : 170 operations, 0 NaN values
discharge : 168 operations, 0 NaN values
impedance : 278 operations, 0 NaN values

B0007
------------------------------
charge    : 170 operations, 0 NaN values
discharge : 168 operations, 0 NaN values
impedance : 278 operations, 0 NaN values

B0018
------------------------------
charge    : 134 operations, 6 NaN values
discharge : 132 operations, 0 NaN values
impedance : 53 operations, 0 NaN values


In [8]:
for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    sample_counts = []

    for i in range(cycle_data.shape[1]):

        cycle_type = cycle_data[0, i]["type"].item()

        if cycle_type == "discharge":

            data = cycle_data[0, i]["data"][0, 0]
            time_values = np.asarray(data["Time"]).flatten()

            sample_counts.append(len(time_values))

    print(f"\n{battery_name}")
    print("-" * 30)
    print("Number of discharge operations:", len(sample_counts))
    print("Minimum samples:", min(sample_counts))
    print("Maximum samples:", max(sample_counts))
    print("Average samples:", round(np.mean(sample_counts), 2))


B0005
------------------------------
Number of discharge operations: 168
Minimum samples: 179
Maximum samples: 371
Average samples: 299.32

B0006
------------------------------
Number of discharge operations: 168
Minimum samples: 179
Maximum samples: 371
Average samples: 299.32

B0007
------------------------------
Number of discharge operations: 168
Minimum samples: 179
Maximum samples: 371
Average samples: 299.32

B0018
------------------------------
Number of discharge operations: 132
Minimum samples: 200
Maximum samples: 366
Average samples: 264.14


In [9]:
capacity_summary = []

for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    capacities = []

    for i in range(cycle_data.shape[1]):

        cycle_type = cycle_data[0, i]["type"].item()

        if cycle_type == "discharge":

            data = cycle_data[0, i]["data"][0, 0]
            capacity = float(data["Capacity"].flatten()[0])

            capacities.append(capacity)

    capacity_summary.append({
        "Battery": battery_name,
        "Discharge Operations": len(capacities),
        "Initial Capacity (Ah)": capacities[0],
        "Final Capacity (Ah)": capacities[-1],
        "Minimum Capacity (Ah)": min(capacities)
    })

capacity_df = pd.DataFrame(capacity_summary)

capacity_df

,Battery,Discharge Operations,Initial Capacity (Ah),Final Capacity (Ah),Minimum Capacity (Ah)
0,B0005,168,1.856487,1.325079,1.287453
1,B0006,168,2.035338,1.185675,1.153818
2,B0007,168,1.891052,1.432455,1.400455
3,B0018,132,1.855005,1.341051,1.341051


In [10]:
for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    duplicate_cycles = 0
    non_monotonic = 0
    empty_time = 0
    total_discharge = 0

    for i in range(cycle_data.shape[1]):

        cycle_type = cycle_data[0, i]["type"].item()

        if cycle_type == "discharge":

            total_discharge += 1

            data = cycle_data[0, i]["data"][0, 0]
            time_values = np.asarray(data["Time"]).flatten()

            if len(time_values) == 0:
                empty_time += 1
                continue

            if len(time_values) != len(np.unique(time_values)):
                duplicate_cycles += 1

            if not np.all(np.diff(time_values) >= 0):
                non_monotonic += 1

    print(f"\n{battery_name}")
    print("-" * 30)
    print("Discharge operations:", total_discharge)
    print("Duplicate timestamp cycles:", duplicate_cycles)
    print("Non-monotonic time sequences:", non_monotonic)
    print("Empty time sequences:", empty_time)


B0005
------------------------------
Discharge operations: 168
Duplicate timestamp cycles: 0
Non-monotonic time sequences: 0
Empty time sequences: 0

B0006
------------------------------
Discharge operations: 168
Duplicate timestamp cycles: 0
Non-monotonic time sequences: 0
Empty time sequences: 0

B0007
------------------------------
Discharge operations: 168
Duplicate timestamp cycles: 0
Non-monotonic time sequences: 0
Empty time sequences: 0

B0018
------------------------------
Discharge operations: 132
Duplicate timestamp cycles: 0
Non-monotonic time sequences: 0
Empty time sequences: 0


## Initial Dataset Findings

- The dataset contains four Li-ion batteries: B0005, B0006, B0007 and B0018.
- Discharge operations are treated as the primary cycle-level records because each discharge operation contains a measured capacity.
- Battery capacity generally decreases with increasing discharge cycle count, indicating degradation.
- Discharge measurements are variable-length time series.
- Voltage, current and temperature measurements are available for each discharge operation.
- B0018 contains a small number of NaN values in charge measurements, while discharge measurements contain no missing values.
- No duplicate timestamps or non-monotonic discharge time sequences were detected.
- The raw MATLAB structure preserves charge, discharge and impedance information for subsequent feature engineering.